In [2]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Stacking Ensemble :

- Stacking is an ensemble machine learning technique where the predictions of multiple base models are used as inputs to train a final model.

- Instead of just voting on a final answer or averaging predictions, a stacking model learns how to combine the predictions of different algorithms intelligently.

### How Stacking Works :

-  First the base models are trained and from there predicted output the Meta Model is trained as the features.

### Problem with the Stacking - It has the tendency to overfit. To overcome the tendency of overfitting we have 2 methods.

1. `Hold Out Approach (Blending)` - Split training data into train + val; train base models on train, generate meta-features on val.
2. `K-Fold Approach` - Use $K-1$ folds to train base models and predict on the remaining fold to build the meta-dataset.

In [4]:
# Generate a synthetic dataset
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define Base Models and  Meta Model
base_models = {
    "rf": RandomForestClassifier(n_estimators=50, random_state=42),
    "gb": GradientBoostingClassifier(n_estimators=50, random_state=42),
    "knn": KNeighborsClassifier(n_neighbors=5)
}
meta_model = LogisticRegression()

# Prepare Out-Of-Fold (OOF) prediction matrix for X_train
n_samples = X_train.shape[0]
n_models = len(base_models)
oof_train_meta = np.zeros((n_samples, n_models))

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Generate Out-Of-Fold Predictions
print("Generating Out-Of-Fold predictions for Meta Model")
for model_idx, (name, model) in enumerate(base_models.items()):
    for train_idx, val_idx in kf.split(X_train):
        # Split folds
        X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
        y_fold_train = y_train[train_idx]
        
        # Train on K-1 folds, predict on the held-out fold
        model.fit(X_fold_train, y_fold_train)
        oof_train_meta[val_idx, model_idx] = model.predict_proba(X_fold_val)[:, 1]

# Train the Meta-Model on OOF Predictions
print("Training the Meta-Model...")
meta_model.fit(oof_train_meta, y_train)

# Retrain Base Models on the Entire Training Set
print("Refitting Base Models on full X_train...")
test_meta_features = np.zeros((X_test.shape[0], n_models))

for model_idx, (name, model) in enumerate(base_models.items()):
    # Fit on all of X_train
    model.fit(X_train, y_train)
    # Generate meta-features for unseen X_test
    test_meta_features[:, model_idx] = model.predict_proba(X_test)[:, 1]

# Final Prediction with Meta-Model
final_preds = meta_model.predict(test_meta_features)
print(f"\nCustom Stacking Accuracy: {accuracy_score(y_test, final_preds):.4f}")

Generating Out-Of-Fold predictions for Meta Model
Training the Meta-Model...
Refitting Base Models on full X_train...

Custom Stacking Accuracy: 0.8950


In [5]:
from sklearn.ensemble import StackingClassifier

# Define the base estimators
estimators = [
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=50, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=5))
]

# Instantiate StackingClassifier 
stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=5,
    stack_method='predict_proba', # Uses probabilities instead of hard classes
    passthrough=False             # Set to True if meta-model should also see original X features
)

# Fit on training data and evaluate
stacking_clf.fit(X_train, y_train)
y_pred = stacking_clf.predict(X_test)

print(f"Scikit-Learn Stacking Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Scikit-Learn Stacking Accuracy: 0.8950
